In [1]:
from typing import Iterator

import os
import gzip
import string
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from datasets import load_from_disk

## Create synthetic dataset

In [2]:
UNITS = {"KB": 10**3, "MB": 10**6, "GB": 10**9}  # 1 base = 1 byte as ASCII

def parse_size(size: str) -> int:
    """'500KB', '1.5MB' or '2GB' -> number of bases."""
    number, unit = size[:-2], size[-2:].upper()
    if unit not in UNITS:
        raise ValueError(f"Invalid size: {size!r}. Use KB, MB or GB, e.g. '500KB', '1.5MB', '2GB'.")
    return int(float(number) * UNITS[unit])


def random_contig(size: str | int, seed: int | None = None) -> dict:
    """size is '500KB'/'1.5MB'/'2GB', or a number of bases."""
    rng = np.random.default_rng(seed)
    n_bases = parse_size(size) if isinstance(size, str) else size

    bases = np.frombuffer(b"ACGT", dtype=np.uint8)
    sequence = bases[rng.integers(0, 4, size=n_bases, dtype=np.uint8)].tobytes().decode("ascii")

    genome_id = f"GCF_{rng.integers(10**9):09d}.1"                                         # e.g. GCF_004512387.1
    contig_id = "".join(rng.choice(list(string.ascii_uppercase), 2)) + f"{rng.integers(10**6):06d}.1"  # e.g. OQ832096.1
    host = rng.choice(["Eco", "Sau", "Pae", "Kpn", "Asp", "Hme"])
    phage = f"vB_{host}{rng.choice(['M', 'P', 'S'])}_{rng.choice(['Kalo', 'Mira', 'Tove', 'Zeno'])}{rng.integers(1, 100)}"

    return {
        "genome_id": genome_id,
        "contig_id": contig_id,
        "sequence": sequence,
        "orientation": "forward",
        "description": f"{contig_id} Synthetic phage {phage}, complete genome",
    }

In [3]:
contig = random_contig("1KB", seed=0)

In [4]:
contig

{'genome_id': 'GCF_272657960.1',
 'contig_id': 'GF888118.1',
 'sequence': 'CGTTTTAGATTGTGACGTTCGACAATCAGAAAACTACCATCACGCCGTCGTGCCCGCGGTCCTGGCTGTAAGTGCGATCTCAACCTTTGGGGCCGATTTCGTCTAGTGCAGACTTTAGTGGAGTCTTAACTAACTTACGATAGGAAGAATGCTTACTAACAGACATAACTCACATAGTAAGAGGGTGGAAGGACTCTTGGCAGTAGACATTCCACTATATCAATACACTGCGATATTCGGGGATGAAGGCAGGCGCGAATAGGAAAAGGCGGACCTTGCGACACTACCAACCTGCCCGAGGCGTGGGATCATGAAGCCGCTGAGCTCGTTACCACCGTAGTAAGCGAGGCGCGGGTGACCTCACAATTGGGCAAAAGATGCAGGTGCAGAGAGAGCGAATCTGCCCCTCTACTCCAAAGTCCCTCTCAGCTAGTTGGTACGGTGAGCAAGATTCCCGCATCTTTCCGGACAAGGCTGCCAGGTGGGGCGAATATTATCCTCGAAAAGGTGGGACATGCCCACTTATAGTATTGCAATAATACATATCAATCCTGTGAGAAGGGGCAAGGGGTAACTGCTCGGGCATCCTACTATAAAATCCTTACTTCGAATACTGCAGCAGGTCTGGATTTCCAAGGTCAGGTGGTTTCCCAACCATGCCAGGTCTACCATCGTATTACCAACGAAGGGTGGCAGCTGTGCGTAACAGACAGCTAAGGGGACTGGCTTTAGTATTTAATTTTAGCGACGATACAAAGATCTTTTGATCAACTCAACACGGATTCCACATTTATCCCAGTGGTAACTCGAGCGTCACAGTGACTTGTATCCTCGGTAAACTCACTGTGGCAGTACCTAAAGACTTTAGTTCAACCTATCGTTTAGGCGGGCGTTCAAACTCTAGAGCCGAAACTCAGAATGAAAT

In [5]:
len(contig["sequence"])

1000

In [6]:
def random_contig_lengths(
    n_contigs: int,
    total_size: str,
    giant_fraction: float = 0.05,
    giant_factor: float = 50.0,
    seed: int | None = None,
) -> np.ndarray:
    """Lengths of n_contigs contigs whose sum is at most total_size.

    Lengths vary lognormally, and giant_fraction of the contigs (at least one) are on
    average giant_factor times longer than the rest.
    """
    total = parse_size(total_size)
    if not 0 < n_contigs <= total:
        raise ValueError(f"Need 0 < n_contigs <= total bases, got {n_contigs} contigs for {total:,} bases.")
    rng = np.random.default_rng(seed)

    weights = rng.lognormal(mean=0.0, sigma=1.0, size=n_contigs)
    n_giants = max(1, round(giant_fraction * n_contigs))
    weights[rng.choice(n_contigs, size=n_giants, replace=False)] *= giant_factor

    # every contig gets at least 1 base; the rest is shared by weight and rounded down,
    # so the sum never exceeds the total
    return 1 + np.floor(weights / weights.sum() * (total - n_contigs)).astype(np.int64)


In [7]:
random_contig_lengths(10, "1MB", seed=0)

array([ 10594,   8186, 886201,  10375,   5468,  13412,  34416,  24085,
         4622,   2636])

In [8]:
def random_contigs(
    n_contigs: int,
    total_size: str,
    giant_fraction: float = 0.05,
    giant_factor: float = 50.0,
    seed: int | None = None,
) -> Iterator[dict]:
    """Yields n_contigs random contigs with a total length of at most total_size.

    Contigs are generated one at a time, so only one sequence is in memory unless you
    keep them, e.g. with list(...).
    """
    rng = np.random.default_rng(seed)
    lengths = random_contig_lengths(n_contigs, total_size, giant_fraction, giant_factor, seed=rng.integers(2**63))
    for length in lengths.tolist():
        yield random_contig(length, seed=rng.integers(2**63))

In [9]:
contigs = list(random_contigs(1000, "100MB", seed=0))

In [10]:
ds = pd.DataFrame(contigs)

In [11]:
ds.head()

,genome_id,contig_id,sequence,orientation,description
0,GCF_096233192.1,EP190191.1,CTAATTGCGGCGCGGTAGCTATGAGGCCAACGGGATATAATGCGCC...,forward,"EP190191.1 Synthetic phage vB_KpnS_Zeno40, com..."
1,GCF_684951774.1,NL179703.1,TACTCTGGCCCGTAATCTGAAATGCAGTGCTTTTTGAAGGTGAAGG...,forward,"NL179703.1 Synthetic phage vB_HmeP_Tove15, com..."
2,GCF_612223867.1,MS008687.1,CGGCATAATTGCCTTAGCCGGTTACCTTCAATTTCACCAAGGAGAG...,forward,"MS008687.1 Synthetic phage vB_AspS_Mira47, com..."
3,GCF_141609484.1,FK296361.1,GCGTTTCTCGTGCGTCACAAACAGGGCGCCCGGGCTTCAGTCAGGC...,forward,"FK296361.1 Synthetic phage vB_PaeM_Tove73, com..."
4,GCF_956603096.1,VB270098.1,ATTCCAGGATTCGGTTTCTCCAGAGGCGATAGTGTTTTATATCGAA...,forward,"VB270098.1 Synthetic phage vB_EcoM_Zeno58, com..."


In [12]:
def save_synthetic_data(df: pd.DataFrame, file_path: str = "synthetic_data.fasta") -> None:
    records = []
    for _, row in df.iterrows():
        records.append(SeqRecord(
            Seq(row["sequence"]),
            id=str(row["contig_id"]),
            description=f"genome_id={row['genome_id']}, orientation={row['orientation']}, {row['description']}",
        ))
    SeqIO.write(records, file_path, "fasta")

### Save different synthetic dataset sizes

In [13]:
base_dir = "/home/datasets/experiments/experiment-bioinfo"

In [14]:
# n, size = 1_000, "100MB"
# contigs = list(random_contigs(n, size, seed=0))
# ds = pd.DataFrame(contigs)
# save_synthetic_data(ds, f"{base_dir}/synthetic_data_{n}_{size}.fasta")

In [15]:
# n, size = 10_000, "1GB"
# contigs = list(random_contigs(n, size, seed=0))
# ds = pd.DataFrame(contigs)
# save_synthetic_data(ds, f"{base_dir}/synthetic_data_{n}_{size}.fasta")

In [16]:
# n, size = 100_000, "10GB"
# contigs = list(random_contigs(n, size, seed=0))
# ds = pd.DataFrame(contigs)
# save_synthetic_data(ds, f"{base_dir}/synthetic_data_{n}_{size}.fasta")

## Stress test

### Sequence dataset

In [17]:
os.environ["PROKBERT_PROFILE"] = "1"

In [18]:
from prokbert.sequence_dataset import SequenceDataset

In [19]:
file_path_100MB = f"{base_dir}/synthetic_data_1000_100MB.fasta"
file_path_1GB = f"{base_dir}/synthetic_data_10000_1GB.fasta"
file_path_10GB = f"{base_dir}/synthetic_data_100000_10GB.fasta"

assert all(os.path.exists(fp) for fp in [file_path_100MB, file_path_1GB, file_path_10GB])


In [20]:
seq_ds = SequenceDataset()

In [21]:
seq_ds.create_dataset([file_path_100MB], save_dir=f"{base_dir}/test_save_dir1")

Loaded contigs from /home/datasets/experiments/experiment-bioinfo/synthetic_data_1000_100MB.fasta, 100.5 MB in 0.56 s -> 178.1 MB/s
Created dataset, 100.6 MB in 0.63 s -> 160.8 MB/s
Saved dataset to '/home/datasets/experiments/experiment-bioinfo/test_save_dir1' (100.4 MB) in 0.28 s -> 362.2 MB/s


In [22]:
seq_ds.load_dataset(f"{base_dir}/test_save_dir1")

Loaded dataset from '/home/datasets/experiments/experiment-bioinfo/test_save_dir1' (100.4 MB) in 0.05 s -> 2175.9 MB/s


In [23]:
seq_ds.create_dataset([file_path_1GB], save_dir=f"{base_dir}/test_save_dir2")

Loaded contigs from /home/datasets/experiments/experiment-bioinfo/synthetic_data_10000_1GB.fasta, 1005.0 MB in 5.08 s -> 197.6 MB/s
Created dataset, 1006.3 MB in 5.83 s -> 172.7 MB/s
Saved dataset to '/home/datasets/experiments/experiment-bioinfo/test_save_dir2' (1003.6 MB) in 2.37 s -> 423.4 MB/s


In [24]:
seq_ds.load_dataset(f"{base_dir}/test_save_dir2")

Loaded dataset from '/home/datasets/experiments/experiment-bioinfo/test_save_dir2' (1003.6 MB) in 0.23 s -> 4456.4 MB/s


In [25]:
seq_ds.create_dataset(
    [file_path_10GB], save_dir=f"{base_dir}/test_save_dir3"
)

Loaded contigs from /home/datasets/experiments/experiment-bioinfo/synthetic_data_100000_10GB.fasta, 10049.7 MB in 48.94 s -> 205.4 MB/s
Created dataset, 10062.8 MB in 55.37 s -> 181.7 MB/s
Saved dataset to '/home/datasets/experiments/experiment-bioinfo/test_save_dir3' (10036.3 MB) in 22.41 s -> 447.9 MB/s


In [26]:
seq_ds.load_dataset(f"{base_dir}/test_save_dir3")

Loaded dataset from '/home/datasets/experiments/experiment-bioinfo/test_save_dir3' (10036.3 MB) in 1.60 s -> 6269.5 MB/s


### Sampler

In [34]:
import importlib
import prokbert.sampler
importlib.reload(prokbert.sampler)

from prokbert.sampler import Sampler

In [35]:
sampler = Sampler(
    sequence_dataset=seq_ds,
    min_segment_length=1_000,
    max_length=2_000,
    segmentation_type="contiguous"
)

In [36]:
os.environ["PROKBERT_PROFILE"] = "1" # enable profiling, return segment metadata

for i, segment in enumerate(iter(sampler)): # iter over full dataset
    if i % 100_000 == 0:
        print(f"Processed {i} segment: {segment}")
    if i > 500_000:
        break

Processed 0 segment: {'segment_id': 0, 'contig_id': 'YV785731.1', 'genome_id': None, 'sequence_id': 0, 'absolute_coordinate': (0, 2000), 'relative_coordinate': (0, 2000), 'orientation': 'forward'}
Processed 100000 segment: {'segment_id': 100000, 'contig_id': 'KO618411.1', 'genome_id': None, 'sequence_id': 2190, 'absolute_coordinate': (200022088, 200024088), 'relative_coordinate': (346000, 348000), 'orientation': 'forward'}
Processed 200000 segment: {'segment_id': 200000, 'contig_id': 'RA406959.1', 'genome_id': None, 'sequence_id': 4425, 'absolute_coordinate': (399986756, 399988756), 'relative_coordinate': (404000, 406000), 'orientation': 'forward'}
Processed 300000 segment: {'segment_id': 300000, 'contig_id': 'QK652971.1', 'genome_id': None, 'sequence_id': 6092, 'absolute_coordinate': (599957096, 599959096), 'relative_coordinate': (5850000, 5852000), 'orientation': 'forward'}
Processed 400000 segment: {'segment_id': 400000, 'contig_id': 'XN181581.1', 'genome_id': None, 'sequence_id': 7

In [37]:
os.environ["PROKBERT_PROFILE"] = "2" # enable profiling, return segment's sequence

for i, segment in enumerate(iter(sampler)): # iter over full dataset
    if i % 100_000 == 0:
        print(f"Processing segment {i}, segment seq: {segment}")
    if i > 500_000:
        break

Processing segment 0, segment seq: CTAATTGCGGCGCGGTAGCTATGAGGCCAACGGGATATAATGCGCCACGGCGTTGAAAAGTATGAGATCAGTTACCCGACGCCGCGGTTTACGACACGAGTAGATGGTCCAGTCCATAAACTATTCTGCTCCTCGGGGGGCGAGCTAGAAACACGGATAAATTCCGGCGGCTAACTGGCACACCCCGACGAGTTGCACAGCAAACGTCTCAAGTTCTTGGTACTATTAGGGATATTTCTCCGACTGACTGCTGTTTACGACCCTACTGTAATCTACTGCATAGTAACGACAAGAGCAACATTCCCCAAGTTGGATCTTGGAAGTCTTTAACATTGTGCAACACGGTCAGCGACACGGCGCGCCCTCACACCACACTCGTTTACTACAATAAATTCACTGAACGTGGGCAGCTATAGCATTTTTCAATAACAGGTCAATGGTCAACTGATAGATTAGCAAATAGCGACACAGTGGTCCAAGCCAGCCTGAGAATAGGAATGCTTGACCACGTGGAGGTGATTTGCTATCTATAATCTTGACACAGTTGTAACACTCCAAAAACGGTCGGGCAGCGGAGCTGACCCCGGATCATCGTTTAATGTGGCGTAAACAAGTTTTCGCCGCCTCACTGTGGAACGCTAGTGATGAGGGTTGCATTCTTAAGTCGCTAATCCCGCGCGACAGCTGACAGGACCGTAGGACTGCGGATGGTTTGTAACTGGTATCATACCCATTAACATATTGAATGTAAGCCTACATACATCGCCGCTAGCCAAGCGTCTTGACACTAGTAAAGTTAAGATTCTCACCACTAAAGTCTGGAATCGTAGAATTTAGCATTTGCGTTGAACGCCACCGTTCAATGACGTAAAAACTTGTCCTTCGTGGTTGACTTGTATCCACGGCCTAGGAAAGCACTAGAAGAGGTATGCACAACCGGTCTAGTTTTTCCTGAGCAATAGTCA

In [39]:
sampler = Sampler(
    sequence_dataset=seq_ds,
    min_segment_length=1_000,
    max_length=2_000,
    segmentation_type="random"
)

In [40]:
os.environ["PROKBERT_PROFILE"] = "1" # enable profiling, return segment

for i, segment in enumerate(iter(sampler)):
    if i % 100_000 == 0:
        print(f"Processed {i} segment: {segment}")
    if i > 1_000_000:
        break

Processed 0 segment: {'segment_id': 0, 'contig_id': 'RK124508.1', 'genome_id': None, 'sequence_id': 91936, 'absolute_coordinate': (9187069008, 9187071008), 'relative_coordinate': (6278338, 6280338), 'orientation': 'forward'}
Processed 100000 segment: {'segment_id': 100000, 'contig_id': 'YK344524.1', 'genome_id': None, 'sequence_id': 77530, 'absolute_coordinate': (7758510627, 7758512627), 'relative_coordinate': (35179, 37179), 'orientation': 'forward'}


Processed 200000 segment: {'segment_id': 200000, 'contig_id': 'BT899257.1', 'genome_id': None, 'sequence_id': 5443, 'absolute_coordinate': (525565590, 525567590), 'relative_coordinate': (7768760, 7770760), 'orientation': 'forward'}
Processed 300000 segment: {'segment_id': 300000, 'contig_id': 'ML693807.1', 'genome_id': None, 'sequence_id': 2667, 'absolute_coordinate': (256074225, 256076225), 'relative_coordinate': (70086, 72086), 'orientation': 'forward'}
Processed 400000 segment: {'segment_id': 400000, 'contig_id': 'QT351335.1', 'genome_id': None, 'sequence_id': 33790, 'absolute_coordinate': (3468389065, 3468391065), 'relative_coordinate': (2265706, 2267706), 'orientation': 'forward'}
Processed 500000 segment: {'segment_id': 500000, 'contig_id': 'SW076899.1', 'genome_id': None, 'sequence_id': 77904, 'absolute_coordinate': (7785073556, 7785075556), 'relative_coordinate': (63128, 65128), 'orientation': 'forward'}
Processed 600000 segment: {'segment_id': 600000, 'contig_id': 'RE827508.1'

In [41]:
os.environ["PROKBERT_PROFILE"] = "2" # enable profiling, return segment's seq
for i, seq in enumerate(iter(sampler)):
    if i % 1_000_000 == 0:
        print(f"Processing segment {i}, segment sequence: {seq}")
    if i > 10_000_000:
        break

Processing segment 0, segment sequence: GCTGGCACTCTAAACGATAGGGGTCTTGAGCGCAGGCCGAGTTATGCATTTGAATCGCTAGAGTCTTCATAGCATCGAATTGATTTACAGCCCTCACCCTTTCATAGGTTCAAGTCTAGGTTAACAAAGGACTCCCCACATGGCTCCGTTACTACGAAGATCACCGCATCGGTGTGCTAGTGCGCATAGGCCGCGATGCGTTCAAATTCTGAGTTGTTTTTAGTGCTGCTATACCGAAGGTAACTATAGAAGCGGGTTCTTCATGTCTCGGACCGCAGTTAAACCCTGGAAGCGCGCGCGTATGGATGTTTACGACATTCTGCGATTGCTGCCTGTGCTCGATAGGATCGACATCACGAAAGTGCAGCCTGGATACAAGTCCGCCAAACTAACCGCAATGAACCATAGTGTACAGAACAATAAGCCGCTTGAACCGCCTACCTTGTATAATCTAGCCGAACTGGGCGCACCACCTACTGAATTCTATTTAGTTCTGCGACGTATTGTTCAGGGAACTTTTCTTGGGGCGCTCACGGCTCACTCAATAATATAGGCCGAAATTCGACTCCTGCACCATTGAGAAAAACGGCAGCTTTTGAGTTTGGAGGAAGCGCGCCTGGATGGGCCGATGACTCGTCGTAACCGGAATGTTTCAAGGTGGCGCTACTGTAAGAGGCTGGAACGACGGGATTGACGATCAGTCATCAAAGATCAGGCTCTCAAACTCAGAAGCTTTAGCCCTTAGCTAAAACACGACTCTCTATCGCGCGGTGCTGCTCACTGGTTGGCGCGGCTACATATCACGCGCACTCCGGAGTGCGAGTCACATTACCGCTACACGTGACGCTGGCCCGAAAATGTTTTAGCCTGCGACATTCGACCGCGATTTTTCCCTAAAGGTGCGGCGTTAGATACACTGGAAAGAGCTGCTGCGCTCCTACTTTAGCAAATGACCCTGTT